# Addis Ababa Ride Demand Forecasting
## 01 — Cleaning and Data Integration (Deliverable A)

This notebook turns the three raw exports (trips, weather, events) into one trustworthy modeling table.

| Section | Deliverable |
|---|---|
| Cleaning log | **A1** |
| Zone / event-type / timestamp standardization + time-zone proof | **A2** |
| Join map | **A3** |
| Join audit | **A4** |
| Join proof | **A5** |
| Feature table | **A6** |
| Integrity checks | **A7** |
| Master tables + data dictionary | **A8** |

**Common clock for every table: Africa/Addis_Ababa local time (EAT, UTC+3, no DST), stored as tz-naive timestamps.**
All statistics (caps, medians, category lists) are learned from the train file only (Rule 8).

## 0. Setup

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
for d in (PROCESSED_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

train_raw = pd.read_csv(RAW_DIR / "ride_demand_train.csv", dtype={"pickup_hour": str, "zone": str})
test_raw = pd.read_csv(RAW_DIR / "ride_demand_test.csv", dtype={"pickup_hour": str, "zone": str})
weather_raw = pd.read_csv(RAW_DIR / "weather_hourly.csv", dtype={"timestamp": str})
events_raw = pd.read_csv(RAW_DIR / "events_calendar.csv", dtype=str)
submission_template = pd.read_csv(RAW_DIR / "submission_template.csv")

for n, d in [("train", train_raw), ("test", test_raw), ("weather", weather_raw),
             ("events", events_raw), ("submission", submission_template)]:
    print(f"{n:<11} {d.shape}")

TZ_LOCAL = "Africa/Addis_Ababa"
TRAIN_START, TRAIN_END = pd.Timestamp("2025-01-01 00:00"), pd.Timestamp("2025-10-31 23:00")
TEST_START, TEST_END = pd.Timestamp("2025-11-01 00:00"), pd.Timestamp("2025-11-14 23:00")

cleaning_log = []
def log_issue(file, columns, issue, n_rows, total, fix, why):
    cleaning_log.append({"file": file, "columns": columns, "issue_type": issue,
                         "rows_affected": int(n_rows), "pct_rows": round(100 * n_rows / total, 2),
                         "fix_applied": fix, "why": why})

train       (85460, 7)
test        (4032, 3)
weather     (7538, 6)
events      (165, 9)
submission  (4032, 2)


## 1. Timestamp parsing (A2b)

Every timestamp column mixes several formats. Instead of letting `pd.to_datetime` guess (which silently reads
`05/11/2025` as 11 May), each format is detected with a regex and parsed with an explicit `format=`.
Day-first is proven below: in the `dd/mm/yyyy` strings the first number exceeds 12 in thousands of rows
while the second never does.

In [2]:
FORMATS = [
    # (label, regex, parser)
    ("iso_offset", r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}[+-]\d{2}:\d{2}$",
     lambda s: pd.to_datetime(s, format="%Y-%m-%dT%H:%M:%S%z", utc=True)),
    ("iso_utc_Z", r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$",
     lambda s: pd.to_datetime(s, format="%Y-%m-%dT%H:%M:%SZ").dt.tz_localize("UTC")),
    ("ymd_hm", r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$",
     lambda s: pd.to_datetime(s, format="%Y-%m-%d %H:%M")),
    ("dmy_hm", r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$",
     lambda s: pd.to_datetime(s, format="%d/%m/%Y %H:%M")),
    ("mon_d_y_ampm", r"^[A-Z][a-z]{2} \d{2}, \d{4} \d{2}:\d{2} [AP]M$",
     lambda s: pd.to_datetime(s, format="%b %d, %Y %I:%M %p")),
]

def parse_mixed(series, naive_tz):
    """Parse a mixed-format text column into tz-naive EAT timestamps.
    naive_tz = clock used by strings that carry no offset ('EAT' or 'UTC')."""
    s = series.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    fmt = pd.Series("unparsed", index=s.index, dtype=object)
    for label, rx, parser in FORMATS:
        m = s.str.match(rx).fillna(False)
        if not m.any():
            continue
        p = parser(s[m])
        if getattr(p.dt, "tz", None) is not None:          # aware -> convert to EAT
            p = p.dt.tz_convert(TZ_LOCAL).dt.tz_localize(None)
        elif naive_tz == "UTC":                            # naive but on UTC clock
            p = p.dt.tz_localize("UTC").dt.tz_convert(TZ_LOCAL).dt.tz_localize(None)
        out[m] = p.astype("datetime64[ns]")
        fmt[m] = label
    return out, fmt

# Day-first proof for dd/mm/yyyy strings
for name, col in [("train", train_raw["pickup_hour"]), ("test", test_raw["pickup_hour"]),
                  ("weather", weather_raw["timestamp"])]:
    sl = col[col.str.contains("/", na=False)].str.split("/")
    print(f"{name:<8} dd/mm rows={len(sl):>6}  first field>12: {(sl.str[0].astype(int) > 12).sum():>6}"
          f"  second field>12: {(sl.str[1].astype(int) > 12).sum()}")

train    dd/mm rows= 25764  first field>12:  15713  second field>12: 0
test     dd/mm rows=  1198  first field>12:    167  second field>12: 0
weather  dd/mm rows=   732  first field>12:    412  second field>12: 0


## 2. Time-zone proof (A2c)

* **Trips**: ISO strings carry `+03:00` → EAT. The naive formats fall in exactly the same range (1 Jan 00:00 – 31 Oct 23:00) and the parsed
  hours form a single clean hourly grid with the ISO rows, so they are EAT too.
* **Weather**: ISO strings end in `Z` → **UTC**. The table starts at `2024-12-30T21:00Z` = 1 Jan 00:00 EAT and the forecast block starts at
  `2025-10-31T21:00Z` = 1 Nov 00:00 EAT — both only make sense if the file is on UTC. Temperature evidence below: on the raw clock the
  daily maximum is at 12:00, i.e. solar noon; after +3 h it peaks at ~15:00 local (normal afternoon maximum for Addis). The `dd/mm` weather rows
  show the same 12:00 peak, so they are also UTC.
* **Events**: no offsets; times like 18:00 kick-offs and 08:00 road closures are local → EAT.

**Conversion applied:** weather `+3 h` (UTC → EAT). Trips and events unchanged.

In [3]:
w_tmp = weather_raw.copy()
w_tmp["fmt"] = np.where(w_tmp["timestamp"].str.contains("T"), "iso_utc_Z", "dmy_hm")
w_tmp["raw_clock"] = pd.to_datetime(w_tmp["timestamp"].str.replace("T", " ").str.replace("Z", ""),
                                    format="mixed", dayfirst=True)
ok = w_tmp["temp_c"].between(-5, 40)
tz_proof = (w_tmp[ok].assign(hour_raw=w_tmp["raw_clock"].dt.hour)
            .pivot_table(index="hour_raw", columns="fmt", values="temp_c", aggfunc="mean").round(1))
tz_proof["hour_after_+3h_EAT"] = (tz_proof.index + 3) % 24
display(tz_proof.T)
print("Raw-clock temperature peak hour:", tz_proof["iso_utc_Z"].idxmax(),
      "-> EAT peak hour:", (tz_proof["iso_utc_Z"].idxmax() + 3) % 24)
print("Weather first raw timestamp:", weather_raw["timestamp"].iloc[0])

hour_raw,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
fmt,,,,,,,,,,,,,,,,,,,,,,,,
dmy_hm,11.3,11.7,12.7,13.3,14.2,16.2,17.0,18.8,19.4,21.0,21.4,22.4,22.1,23.0,21.6,20.8,19.9,18.7,17.0,15.7,14.8,13.4,12.0,12.1
iso_utc_Z,11.6,11.8,12.3,13.3,14.4,15.7,17.2,18.5,19.8,20.9,21.7,22.1,22.3,21.9,21.5,20.6,19.4,18.2,17.0,15.6,14.3,13.3,12.4,11.8
hour_after_+3h_EAT,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,0.0,1.0,2.0


Raw-clock temperature peak hour: 12 -> EAT peak hour: 15
Weather first raw timestamp: 2024-12-30T21:00:00Z


## 3. Zone and event-type standardization (A2a)

In [4]:
CANONICAL_ZONES = ["Arat Kilo", "Ayat", "Bole", "CMC", "Gerji", "Kazanchis",
                   "Kolfe", "Lideta", "Megenagna", "Merkato", "Piassa", "Sarbet"]
ZONE_ALIASES = {  # spelling variants seen in the raw files -> canonical label
    "bole rd": "Bole", "mercato": "Merkato", "kazanches": "Kazanchis", "megenaga": "Megenagna",
    "piazza": "Piassa", "kolfe keranio": "Kolfe", "c.m.c": "CMC", "cmc": "CMC",
}
CITYWIDE = {"citywide", "city-wide", "all", "all zones"}
_canon_lower = {z.lower(): z for z in CANONICAL_ZONES}

def clean_zone(value):
    if pd.isna(value):
        return np.nan
    v = re.sub(r"\s+", " ", str(value)).strip().lower()
    v = re.sub(r"\s*\(.*?\)", "", v).strip()          # drop '(Kirkos)', '(Bole subcity)'
    v = ZONE_ALIASES.get(v, v)
    return _canon_lower.get(str(v).lower(), np.nan)

def clean_event_zones(value):
    """Events may name several zones ('Lideta & KAZANCHIS') or the whole city."""
    if pd.isna(value):
        return []
    v = re.sub(r"\s+", " ", str(value)).strip().lower()
    if v in CITYWIDE:
        return list(CANONICAL_ZONES)
    return [z for z in (clean_zone(p) for p in re.split(r"&|,|/| and ", v)) if isinstance(z, str)]

EVENT_TYPES = ["public_holiday", "school_break", "football_match", "concert",
               "conference", "exhibition", "road_closure", "sports_run"]
def clean_event_type(v):
    t = re.sub(r"[\s\-]+", "_", str(v).strip().lower())
    return t if t in EVENT_TYPES else np.nan

zone_before_after = pd.concat([
    pd.DataFrame({"table": t, "raw_value": s.value_counts(dropna=False).index,
                  "rows": s.value_counts(dropna=False).values})
    for t, s in [("train", train_raw["zone"]), ("test", test_raw["zone"]), ("events", events_raw["zone"])]
])
zone_before_after["clean_value"] = [
    ", ".join(clean_event_zones(v)) if t == "events" else clean_zone(v)
    for t, v in zip(zone_before_after["table"], zone_before_after["raw_value"])]
display(zone_before_after.sort_values(["table", "clean_value"]).reset_index(drop=True))

et = events_raw["event_type"].value_counts(dropna=False).rename_axis("raw_value").reset_index(name="rows")
et["clean_value"] = et["raw_value"].map(clean_event_type)
display(et)
print("Raw distinct zone labels  train/test/events:",
      train_raw.zone.nunique(), test_raw.zone.nunique(), events_raw.zone.nunique())

,table,raw_value,rows,clean_value
0,events,arat kilo,1,Arat Kilo
1,events,Citywide,9,"Arat Kilo, Ayat, Bole, CMC, Gerji, Kazanchis, ..."
2,events,ALL,3,"Arat Kilo, Ayat, Bole, CMC, Gerji, Kazanchis, ..."
3,events,All zones,2,"Arat Kilo, Ayat, Bole, CMC, Gerji, Kazanchis, ..."
4,events,city-wide,1,"Arat Kilo, Ayat, Bole, CMC, Gerji, Kazanchis, ..."
...,...,...,...,...
133,train,Piazza,639,Piassa
134,train,Sarbet,4446,Sarbet
135,train,Sarbet,945,Sarbet
136,train,sarbet,939,Sarbet


,raw_value,rows,clean_value
0,football_match,22,football_match
1,conference,19,conference
2,Conference,19,conference
3,Football Match,15,football_match
4,football match,14,football_match
5,CONCERT,14,concert
6,road closure,11,road_closure
7,road_closure,9,road_closure
8,concert,8,concert
9,public_holiday,6,public_holiday


Raw distinct zone labels  train/test/events: 55 55 28


## 4. Clean the trip table

In [5]:
N = len(train_raw)
trips = train_raw.copy()
trips["datetime"], trips["ts_format"] = parse_mixed(trips["pickup_hour"], naive_tz="EAT")
display(trips["ts_format"].value_counts().rename("rows").to_frame())
assert trips["datetime"].notna().all()
log_issue("ride_demand_train", "pickup_hour", "mixed timestamp formats (3)", (trips.ts_format != "ymd_hm").sum(), N,
          "regex-detected explicit formats; +03:00 converted to EAT", "avoid day/month swaps and clock drift")

trips["zone_raw"] = trips["zone"]
trips["zone"] = trips["zone_raw"].map(clean_zone)
assert trips["zone"].notna().all()
log_issue("ride_demand_train", "zone", "inconsistent zone spelling (55 variants)",
          (trips.zone_raw != trips.zone).sum(), N, "case/space normalization + alias map to 12 labels",
          "one key per zone across all tables")

# Duplicate zone-hours (same record entered twice under a new record_id)
dup = trips.duplicated(subset=["zone", "datetime"], keep="first")
log_issue("ride_demand_train", "zone, pickup_hour", "duplicate zone-hour rows", dup.sum(), N,
          "kept first occurrence", "exactly one row per zone-hour")
trips = trips[~dup].copy()

# Sentinel -1 in trips
sent = trips["trips"].eq(-1)
log_issue("ride_demand_train", "trips", "sentinel -1 (no reading)", sent.sum(), N,
          "set to NaN, row excluded from training target", "-1 trips is impossible")
trips.loc[sent, "trips"] = np.nan
log_issue("ride_demand_train", "trips", "missing target", trips["trips"].isna().sum() - sent.sum(), N,
          "row excluded from training target (kept in grid for lags)", "cannot learn from unknown target")

# Impossible spikes: trips far above what the active drivers could serve (~1.3 trips/driver-hour)
ratio = trips["trips"] / trips["active_drivers"]
SPIKE_RATIO = 4.0      # >4 trips per active driver in one hour is not physically plausible
spike = ratio > SPIKE_RATIO
print("Typical trips/driver ratio (median):", round(ratio.median(), 2))
display(trips.loc[spike, ["zone", "datetime", "trips", "active_drivers"]].head(8))
log_issue("ride_demand_train", "trips", "data-entry spikes (trips > 4x active drivers)", spike.sum(), N,
          "set to NaN", "e.g. 640 trips with 56 drivers; real event peaks keep the driver ratio")
trips.loc[spike, "trips"] = np.nan

wait_bad = trips["avg_wait_min"].lt(0)
log_issue("ride_demand_train", "avg_wait_min", "sentinel -1", wait_bad.sum(), N, "set to NaN",
          "negative wait impossible (analysis-only column)")
trips.loc[wait_bad, "avg_wait_min"] = np.nan
log_issue("ride_demand_train", "avg_fare_birr", "missing values", trips["avg_fare_birr"].isna().sum(), N,
          "left NaN (not a model input)", "operational column, analysis only")

,rows
ts_format,
ymd_hm,46886
dmy_hm,25764
iso_offset,12810


Typical trips/driver ratio (median): 1.31


,zone,datetime,trips,active_drivers
406,Lideta,2025-06-05 14:00:00,240.0,23
2098,Piassa,2025-05-24 01:00:00,48.0,6
2651,Megenagna,2025-04-09 08:00:00,608.0,43
2659,Megenagna,2025-09-10 20:00:00,640.0,56
2733,Kolfe,2025-09-27 18:00:00,376.0,29
3503,Piassa,2025-07-27 23:00:00,120.0,13
3639,Gerji,2025-04-27 14:00:00,304.0,31
4758,Bole,2025-02-04 14:00:00,192.0,15


### Gaps: outage vs late launch vs random missing (feeds B4.2)

In [6]:
launch = trips.groupby("zone")["datetime"].min().rename("first_hour")
display(launch.sort_values().to_frame())

city_rows = trips.groupby(trips["datetime"].dt.date).size()
outage_days = city_rows[city_rows < 0.5 * city_rows.median()]
print("Days with <50% of normal rows (platform outage):"); display(outage_days.to_frame("rows"))

full_hours = pd.date_range(TRAIN_START, TRAIN_END, freq="h")
grid = pd.MultiIndex.from_product([CANONICAL_ZONES, full_hours], names=["zone", "datetime"]).to_frame(index=False)
grid = grid.merge(launch.reset_index(), on="zone")
grid["pre_launch"] = grid["datetime"] < grid["first_hour"]
hour_rows = trips.groupby("datetime").size()
outage_hours = hour_rows.reindex(full_hours, fill_value=0)
outage_hours = outage_hours[outage_hours == 0].index
grid["outage"] = grid["datetime"].isin(outage_hours)
present = grid.merge(trips[["zone", "datetime"]].assign(present=True), how="left", on=["zone", "datetime"])["present"].fillna(False)
grid["missing_random"] = ~present & ~grid["pre_launch"] & ~grid["outage"]
gap_summary = pd.Series({
    "pre-launch zone-hours (Ayat before launch)": int(grid.pre_launch.sum()),
    "platform-outage zone-hours": int((grid.outage & ~grid.pre_launch).sum()),
    "random missing zone-hours": int(grid.missing_random.sum())})
display(gap_summary.to_frame("zone_hours"))
log_issue("ride_demand_train", "(rows)", "platform outage – whole city missing", len(outage_hours), len(full_hours),
          "not imputed; hours excluded from training", "no demand was recorded, imputing would invent data")
log_issue("ride_demand_train", "(rows)", "late-launching zone (Ayat) – no rows before launch",
          grid.pre_launch.sum(), len(grid), "zone-hours before launch excluded",
          "zone did not exist; not zero demand")

,first_hour
zone,
Arat Kilo,2025-01-01
Bole,2025-01-01
CMC,2025-01-01
Gerji,2025-01-01
Kazanchis,2025-01-01
Kolfe,2025-01-01
Lideta,2025-01-01
Megenagna,2025-01-01
Merkato,2025-01-01


Days with <50% of normal rows (platform outage):


,rows
datetime,
2025-05-12,24
2025-05-13,48


,zone_hours
pre-launch zone-hours (Ayat before launch),1752
platform-outage zone-hours,504
random missing zone-hours,682


## 5. Clean the weather table

In [7]:
NW = len(weather_raw)
w = weather_raw.copy()
w["datetime"], w["ts_format"] = parse_mixed(w["timestamp"], naive_tz="UTC")
assert w["datetime"].notna().all()
log_issue("weather_hourly", "timestamp", "UTC clock + 2 formats", NW, NW,
          "parsed explicitly, converted UTC -> EAT (+3h)", "trip table is on EAT (see A2c proof)")

w["data_type"] = w["data_type"].str.strip().str.lower()

sent = w["rain_mm"].eq(-9999)
log_issue("weather_hourly", "rain_mm", "sentinel -9999", sent.sum(), NW, "set to NaN, later filled",
          "sentinel code for no reading")
w.loc[sent, "rain_mm"] = np.nan

# Fahrenheit block: a contiguous mid-July run where temperature is ~60-70
fahr = w["temp_c"] > 40
print("Rows >40 'C by date:", w.loc[fahr, "datetime"].dt.date.min(), "->", w.loc[fahr, "datetime"].dt.date.max())
log_issue("weather_hourly", "temp_c", "values in Fahrenheit for part of July", fahr.sum(), NW,
          "converted (F-32)*5/9", "40-77 'C is impossible in Addis; converted values match neighbours")
w.loc[fahr, "temp_c"] = (w.loc[fahr, "temp_c"] - 32) * 5 / 9

d = w.duplicated(subset=["datetime"], keep=False)
log_issue("weather_hourly", "timestamp", "duplicate hours with conflicting values", w.duplicated(subset=["datetime"]).sum(), NW,
          "averaged numeric values per hour", "many-to-one join needs one row per hour")
num = ["temp_c", "rain_mm", "humidity_pct", "wind_kmh"]
w = w.groupby("datetime", as_index=False).agg({**{c: "mean" for c in num}, "data_type": "first"})

full_w = pd.date_range(TRAIN_START, TEST_END, freq="h")
missing_hours = full_w.difference(w["datetime"])
log_issue("weather_hourly", "timestamp", "missing hours (gaps)", len(missing_hours), len(full_w),
          "reindexed to full hourly grid; interpolated (≤6h), rain filled 0", "every zone-hour needs weather")
log_issue("weather_hourly", "temp_c, humidity_pct", "missing values",
          w[["temp_c", "humidity_pct"]].isna().any(axis=1).sum(), NW, "time interpolation", "smooth variables")
w = w.set_index("datetime").reindex(full_w)
w.index.name = "datetime"
w["weather_imputed"] = w["temp_c"].isna() | w["rain_mm"].isna()
for c in ["temp_c", "humidity_pct", "wind_kmh"]:
    w[c] = w[c].interpolate(method="time", limit=6, limit_direction="both")
# Long gaps (e.g. a 19 h outage on 3 Sep): fill with the hour-of-day mean learned from observed history only
hist = w.index < TEST_START
long_gap = w[["temp_c", "humidity_pct", "wind_kmh"]].isna().any(axis=1)
print("Hours in long weather gaps (filled with hour-of-day climatology):", int(long_gap.sum()))
for c in ["temp_c", "humidity_pct", "wind_kmh"]:
    clim = w.loc[hist, c].groupby(w.index[hist].hour).mean()
    w[c] = w[c].fillna(pd.Series(w.index.hour.map(clim), index=w.index))
w["rain_mm"] = w["rain_mm"].fillna(0.0)          # dry is by far the most common state
w["data_type"] = w["data_type"].fillna(pd.Series(np.where(w.index >= TEST_START, "forecast", "observed"), index=w.index))
w = w.reset_index()

# Weather features
w["rain_3h_sum"] = w["rain_mm"].rolling(3, min_periods=1).sum()
w["is_raining"] = (w["rain_mm"] > 0).astype(int)
w["rain_class"] = pd.cut(w["rain_mm"], [-0.01, 0, 2.5, 7.6, np.inf],
                         labels=["none", "light", "moderate", "heavy"]).astype(str)
weather_clean = w
display(weather_clean.describe().T.round(2))

Rows >40 'C by date: 2025-07-10 -> 2025-07-25
Hours in long weather gaps (filled with hour-of-day climatology): 19


,count,mean,min,25%,50%,75%,max,std
datetime,7632,2025-06-08 23:30:00,2025-01-01 00:00:00,2025-03-21 11:45:00,2025-06-08 23:30:00,2025-08-27 11:15:00,2025-11-14 23:00:00,NaN
temp_c,7632.0,17.043458,5.4,13.7,16.9,20.391667,28.0,4.250901
rain_mm,7632.0,0.240691,0.0,0.0,0.0,0.0,25.7,1.280538
humidity_pct,7632.0,72.833055,37.0,64.0,73.5,82.0,100.0,11.593713
wind_kmh,7632.0,10.884905,3.1,7.8,10.4,13.4,31.0,4.11172
rain_3h_sum,7632.0,0.722072,0.0,0.0,0.0,0.0,32.3,2.516778
is_raining,7632.0,0.074292,0.0,0.0,0.0,0.0,1.0,0.262263


## 6. Clean the events table

In [8]:
NE = len(events_raw)
ev = events_raw.copy()
ev["event_type_clean"] = ev["event_type"].map(clean_event_type)
log_issue("events_calendar", "event_type", "inconsistent spelling (20 variants)",
          (ev.event_type != ev.event_type_clean).sum(), NE, "normalized to 8 labels", "group events by type")

ev["status"] = ev["status"].str.strip().str.lower()
log_issue("events_calendar", "status", "case/whitespace variants", (events_raw.status != ev.status).sum(), NE,
          "lower + strip", "reliable cancelled filter")

ev["zones"] = ev["zone"].map(clean_event_zones)
log_issue("events_calendar", "zone", "spelling, extra text, multi-zone and city-wide values",
          (ev["zone"].map(clean_zone) != ev["zone"]).sum(), NE,
          "split on '&', strip '(...)', alias map, citywide -> all 12", "interval join needs canonical zone keys")

ev["start"], ev["start_fmt"] = parse_mixed(ev["start_datetime"], naive_tz="EAT")
ev["end"], ev["end_fmt"] = parse_mixed(ev["end_datetime"], naive_tz="EAT")
assert ev["start"].notna().all()
log_issue("events_calendar", "start_datetime, end_datetime", "3 timestamp formats incl. 'Feb 06, 2025 08:00 PM'",
          (ev.start_fmt != "ymd_hm").sum() + (ev.end_fmt != "ymd_hm").sum(), 2 * NE,
          "explicit regex formats", "no day/month confusion")

med_dur = (ev["end"] - ev["start"])[ev["end"] > ev["start"]].groupby(ev["event_type_clean"]).median()
bad_end = ev["end"].isna() | (ev["end"] <= ev["start"])
log_issue("events_calendar", "end_datetime", "missing or earlier than start", bad_end.sum(), NE,
          "replaced by start + median duration of that event type", "interval needs a valid end")
ev.loc[bad_end, "end"] = ev.loc[bad_end, "start"] + ev.loc[bad_end, "event_type_clean"].map(med_dur)

att = (ev["expected_attendance"].str.lower().str.replace("approx", "", regex=False)
       .str.replace(",", "", regex=False).str.strip())
ev["attendance"] = pd.to_numeric(att, errors="coerce")
log_issue("events_calendar", "expected_attendance", "free text ('approx 34000', '34,756') and blanks",
          ev["expected_attendance"].notna().sum(), NE, "parsed to number; blanks -> 0 in features",
          "usable numeric crowd size")
log_issue("events_calendar", "status", "cancelled events", ev.status.eq("cancelled").sum(), NE,
          "excluded from event features (kept in file for B3.4)", "did not take place")

events_clean = ev[["event_id", "event_name", "event_type_clean", "venue", "zone", "zones", "start", "end",
                   "attendance", "status"]].rename(columns={"event_type_clean": "event_type"})
display(events_clean.head())

,event_id,event_name,event_type,venue,zone,zones,start,end,attendance,status
0,EVT-0014,Premier league match,football_match,Addis Ababa Stadium,Kazanchis (Kirkos),[Kazanchis],2025-02-06 18:00:00,2025-02-06 20:00:00,34000.0,confirmed
1,EVT-0081,Road closure (construction),road_closure,NaN,AYAT,[Ayat],2025-06-07 08:00:00,2025-06-07 18:00:00,NaN,confirmed
2,EVT-0106,Premier league match,football_match,Addis Ababa Stadium,Kazanchis,[Kazanchis],2025-08-13 19:00:00,2025-08-13 21:00:00,30930.0,confirmed
3,EVT-0069,Road closure (construction),road_closure,NaN,cmc,[CMC],2025-05-14 13:00:00,2025-05-15 03:00:00,NaN,confirmed
4,EVT-0045,Good Friday,public_holiday,NaN,Citywide,"[Arat Kilo, Ayat, Bole, CMC, Gerji, Kazanchis,...",2025-04-18 00:00:00,2025-04-18 23:59:00,NaN,confirmed


## 7. Join map (A3)

```
                 weather_clean (1 row / hour, EAT)
                          │  key: datetime   many-to-one, LEFT
                          ▼
 zone-hour grid  ◄──── trips_clean (left table)            ──►  master_train
 (zone, datetime)         ▲
                          │  key: (zone, datetime ∈ [start-2h, end+2h])   interval join, aggregated to 1 row per zone-hour, LEFT
                 events_clean (exploded to zone × hour)
```

* **Left table = trip zone-hours** (train) / **test rows** (test): the unit we predict; a left join guarantees the row count never changes.
* **Weather**: many zone-hours → one weather hour; `validate="m:1"`.
* **Events**: each confirmed event is exploded to every hour from **2 h before start to 2 h after end** in each affected zone (city-wide → all 12).
  Overlapping events are aggregated (count, max attendance, any-of-type flags) so the join is again many-to-one.
  Public holidays / school breaks are treated as whole-day calendar flags (city-wide).

In [9]:
PRE_H, POST_H = 2, 2
WINDOW_TYPES = ["football_match", "concert", "conference", "exhibition", "road_closure", "sports_run"]
active_ev = events_clean[events_clean["status"] != "cancelled"]

rows = []
for _, e in active_ev[active_ev.event_type.isin(WINDOW_TYPES)].iterrows():
    s, en = e["start"].floor("h"), e["end"].ceil("h")
    for h in pd.date_range(s - pd.Timedelta(hours=PRE_H), en + pd.Timedelta(hours=POST_H), freq="h"):
        phase = "pre" if h < s else ("post" if h >= en else "during")
        for z in e["zones"]:
            rows.append((z, h, e["event_id"], e["event_type"], phase, e["attendance"],
                         (s - h) / pd.Timedelta(hours=1), (h - en) / pd.Timedelta(hours=1)))
ev_hours = pd.DataFrame(rows, columns=["zone", "datetime", "event_id", "event_type", "phase",
                                       "attendance", "hours_to_start", "hours_since_end"])

ev_feat = ev_hours.groupby(["zone", "datetime"]).agg(
    event_in_window=("event_id", "size"),
    event_attendance=("attendance", "max"),
    football_window=("event_type", lambda s: int((s == "football_match").any())),
    concert_window=("event_type", lambda s: int((s == "concert").any())),
    conference_window=("event_type", lambda s: int((s == "conference").any())),
    road_closure_window=("event_type", lambda s: int((s == "road_closure").any())),
    event_pre=("phase", lambda s: int((s == "pre").any())),
    event_during=("phase", lambda s: int((s == "during").any())),
    event_post=("phase", lambda s: int((s == "post").any())),
).reset_index()
ev_feat["event_in_window"] = (ev_feat["event_in_window"] > 0).astype(int)
ev_feat["event_attendance"] = ev_feat["event_attendance"].fillna(0)

def day_set(types):
    days = set()
    for _, e in active_ev[active_ev.event_type.isin(types)].iterrows():
        days.update(pd.date_range(e["start"].normalize(), e["end"].normalize(), freq="D"))
    return days
HOLIDAY_DAYS = day_set(["public_holiday"])
SCHOOL_BREAK_DAYS = day_set(["school_break"])
print("Exploded event zone-hours:", len(ev_hours), "| unique zone-hours with an event:", len(ev_feat))
print("Holiday days:", sorted(d.date() for d in HOLIDAY_DAYS))

Exploded event zone-hours: 2338 | unique zone-hours with an event: 2082
Holiday days: [datetime.date(2025, 1, 7), datetime.date(2025, 1, 19), datetime.date(2025, 3, 2), datetime.date(2025, 3, 31), datetime.date(2025, 4, 18), datetime.date(2025, 4, 20), datetime.date(2025, 5, 1), datetime.date(2025, 5, 5), datetime.date(2025, 5, 28), datetime.date(2025, 6, 6), datetime.date(2025, 9, 4), datetime.date(2025, 9, 11), datetime.date(2025, 9, 27)]


## 8. Feature engineering and joins (A6)

In [10]:
SERIES_START = TRAIN_START

def add_calendar(df):
    t = df["datetime"]
    df["hour"] = t.dt.hour
    df["day_of_week"] = t.dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["month"] = t.dt.month
    df["day_of_month"] = t.dt.day
    df["is_payday_window"] = ((t.dt.day >= 25) | (t.dt.day <= 3)).astype(int)
    df["is_public_holiday"] = t.dt.normalize().isin(HOLIDAY_DAYS).astype(int)
    df["is_school_break"] = t.dt.normalize().isin(SCHOOL_BREAK_DAYS).astype(int)
    df["days_since_start"] = (t - SERIES_START) / pd.Timedelta(days=1)   # trend
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    return df

WEATHER_FEATS = ["temp_c", "rain_mm", "humidity_pct", "wind_kmh", "rain_3h_sum", "is_raining", "rain_class"]
EVENT_FEATS = ["event_in_window", "event_attendance", "football_window", "concert_window",
               "conference_window", "road_closure_window", "event_pre", "event_during", "event_post"]

def join_weather_events(df):
    n0 = len(df)
    df = df.merge(weather_clean[["datetime"] + WEATHER_FEATS], on="datetime", how="left", validate="m:1")
    df = df.merge(ev_feat, on=["zone", "datetime"], how="left", validate="m:1")
    df[EVENT_FEATS] = df[EVENT_FEATS].fillna(0)
    assert len(df) == n0, "row count changed by join"
    return df

# --- Lag features: forecast horizon is up to 14 days (336 h), so only lags >= 336 h are known at forecast time.
lag_base = grid.loc[~grid.pre_launch, ["zone", "datetime"]].merge(
    trips[["zone", "datetime", "trips"]], how="left", on=["zone", "datetime"])
test_keys = test_raw.assign(datetime=parse_mixed(test_raw["pickup_hour"], "EAT")[0],
                            zone=test_raw["zone"].map(clean_zone))[["zone", "datetime"]]
lag_frame = pd.concat([lag_base, test_keys.assign(trips=np.nan)], ignore_index=True)
lag_frame = lag_frame.sort_values(["zone", "datetime"]).reset_index(drop=True)
g = lag_frame.groupby("zone")["trips"]
lag_frame["lag_336h"] = g.shift(336)          # same hour, 2 weeks earlier
lag_frame["lag_504h"] = g.shift(504)          # same hour, 3 weeks earlier
lag_frame["roll_mean_168h_lag336"] = g.transform(lambda s: s.shift(336).rolling(168, min_periods=24).mean())
LAG_FEATS = ["lag_336h", "lag_504h", "roll_mean_168h_lag336"]

In [11]:
# ---------- master_train ----------
mt = trips.loc[trips["trips"].notna(),
               ["record_id", "zone", "datetime", "trips", "avg_fare_birr", "avg_wait_min", "active_drivers"]].copy()
mt = mt.merge(launch.reset_index(), on="zone")
mt = mt[mt["datetime"] >= mt.pop("first_hour")]
n_before_weather = len(mt)
mt = add_calendar(mt)
mt = join_weather_events(mt)
mt = mt.merge(lag_frame[["zone", "datetime"] + LAG_FEATS], on=["zone", "datetime"], how="left", validate="1:1")
master_train = mt.sort_values(["datetime", "zone"]).reset_index(drop=True)

# ---------- master_test (original row order kept) ----------
te = test_raw.copy()
te["datetime"], _ = parse_mixed(te["pickup_hour"], naive_tz="EAT")
te["zone"] = te["zone"].map(clean_zone)
te = add_calendar(te)
te = join_weather_events(te)
te = te.merge(lag_frame[["zone", "datetime"] + LAG_FEATS], on=["zone", "datetime"], how="left", validate="1:1")
master_test = te[["row_id", "pickup_hour", "zone", "datetime"] +
                 [c for c in master_train.columns if c not in
                  ["record_id", "zone", "datetime", "trips", "avg_fare_birr", "avg_wait_min", "active_drivers"]]]
print("master_train", master_train.shape, "| master_test", master_test.shape)
display(master_train.head())

master_train (82979, 37) | master_test (4032, 34)


,record_id,zone,datetime,trips,avg_fare_birr,avg_wait_min,active_drivers,hour,day_of_week,is_weekend,month,day_of_month,is_payday_window,is_public_holiday,is_school_break,days_since_start,hour_sin,hour_cos,temp_c,rain_mm,humidity_pct,wind_kmh,rain_3h_sum,is_raining,rain_class,event_in_window,event_attendance,football_window,concert_window,conference_window,road_closure_window,event_pre,event_during,event_post,lag_336h,lag_504h,roll_mean_168h_lag336
0,R-070502,Arat Kilo,2025-01-01,5.0,240.4,1.9,5,0,2,0,1,1,1,0,0,0.0,0.0,1.0,9.8,0.0,89.0,16.8,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,R-058374,Bole,2025-01-01,8.0,322.8,3.0,6,0,2,0,1,1,1,0,0,0.0,0.0,1.0,9.8,0.0,89.0,16.8,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2,R-073684,CMC,2025-01-01,4.0,289.2,1.6,5,0,2,0,1,1,1,0,0,0.0,0.0,1.0,9.8,0.0,89.0,16.8,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
3,R-016735,Gerji,2025-01-01,2.0,290.2,1.5,3,0,2,0,1,1,1,0,0,0.0,0.0,1.0,9.8,0.0,89.0,16.8,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,R-002178,Kazanchis,2025-01-01,0.0,248.5,1.2,2,0,2,0,1,1,1,0,0,0.0,0.0,1.0,9.8,0.0,89.0,16.8,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN


## 9. Join audit (A4)

In [12]:
audit = pd.DataFrame([
    {"join": "train ⟵ weather (m:1)", "rows_before": n_before_weather, "rows_after": len(master_train),
     "match_rate_%": round(100 * master_train["temp_c"].notna().mean(), 2),
     "zone_hours_with_imputed_weather": int(master_train["datetime"].isin(
         weather_clean.loc[weather_clean.weather_imputed, "datetime"]).sum())},
    {"join": "test ⟵ weather (m:1)", "rows_before": len(test_raw), "rows_after": len(master_test),
     "match_rate_%": round(100 * master_test["temp_c"].notna().mean(), 2),
     "zone_hours_with_imputed_weather": int(master_test["datetime"].isin(
         weather_clean.loc[weather_clean.weather_imputed, "datetime"]).sum())},
])
display(audit)
print("Imputed weather = hours missing/sentinel in the raw file, filled by time interpolation (rain -> 0).")

hrs_train = set(zip(master_train.zone, master_train.datetime))
hrs_test = set(zip(master_test.zone, master_test.datetime))
ev_match = ev_hours.assign(k=list(zip(ev_hours.zone, ev_hours.datetime)))
ev_match["in_train"] = ev_match.k.isin(hrs_train)
ev_match["in_test"] = ev_match.k.isin(hrs_test)
per_event = ev_match.groupby("event_id")[["in_train", "in_test"]].any()
cal = events_clean.set_index("event_id")
summary = {
    "events in file": len(events_clean),
    "cancelled (excluded)": int(cal.status.eq("cancelled").sum()),
    "holiday / school break (day-level calendar flag)": int(cal.event_type.isin(["public_holiday", "school_break"]).sum()),
    "window events matched >=1 train zone-hour": int(per_event.in_train.sum()),
    "window events matched >=1 test zone-hour": int(per_event.in_test.sum()),
    "window events matching no zone-hour (outage / pre-launch)": int((~per_event.any(axis=1)).sum()),
}
display(pd.Series(summary).to_frame("count"))
print("Train zone-hours inside an event window:", int(master_train.event_in_window.sum()),
      f"({100 * master_train.event_in_window.mean():.2f}%)")

,join,rows_before,rows_after,match_rate_%,zone_hours_with_imputed_weather
0,train ⟵ weather (m:1),82979,82979,100.0,3447
1,test ⟵ weather (m:1),4032,4032,100.0,96


Imputed weather = hours missing/sentinel in the raw file, filled by time interpolation (rain -> 0).


,count
events in file,165
cancelled (excluded),6
holiday / school break (day-level calendar flag),15
window events matched >=1 train zone-hour,136
window events matched >=1 test zone-hour,8
window events matching no zone-hour (outage / pre-launch),0


Train zone-hours inside an event window: 1956 (2.36%)


## 10. Join proof (A5)

In [13]:
show = ["zone", "datetime", "trips"] + WEATHER_FEATS + EVENT_FEATS + ["is_public_holiday"]
rain_row = master_train.loc[master_train["rain_mm"].idxmax()]
foot_row = master_train[master_train.football_window.eq(1) & master_train.event_during.eq(1)].iloc[0]
hol_row = master_train[master_train.is_public_holiday.eq(1)].iloc[0]
for label, r in [("Rainy zone-hour", rain_row), ("Football window", foot_row), ("Public holiday", hol_row)]:
    print("=" * 30, label, "=" * 30)
    print("Weather row attached:")
    display(weather_clean[weather_clean.datetime.eq(r.datetime)])
    print("Event rows attached:")
    display(ev_hours[(ev_hours.zone == r.zone) & (ev_hours.datetime == r.datetime)]
            .merge(events_clean[["event_id", "event_name", "start", "end"]], on="event_id"))
    print("Resulting feature values:")
    display(r[show].to_frame().T)

============================== Rainy zone-hour ==============================
Weather row attached:


,datetime,temp_c,rain_mm,humidity_pct,wind_kmh,data_type,weather_imputed,rain_3h_sum,is_raining,rain_class
2921,2025-05-02 17:00:00,21.5,21.0,76.0,6.1,observed,False,21.8,1,heavy


Event rows attached:


,zone,datetime,event_id,event_type,phase,attendance,hours_to_start,hours_since_end,event_name,start,end


Resulting feature values:


...

,datetime,temp_c,rain_mm,humidity_pct,wind_kmh,data_type,weather_imputed,rain_3h_sum,is_raining,rain_class
144,2025-01-07,8.7,0.0,94.333333,10.0,observed,False,0.0,0,none


Event rows attached:


,zone,datetime,event_id,event_type,phase,attendance,hours_to_start,hours_since_end,event_name,start,end


Resulting feature values:


,zone,datetime,trips,temp_c,rain_mm,humidity_pct,wind_kmh,rain_3h_sum,is_raining,rain_class,event_in_window,event_attendance,football_window,concert_window,conference_window,road_closure_window,event_pre,event_during,event_post,is_public_holiday
1542,Arat Kilo,2025-01-07 00:00:00,4.0,8.7,0.0,94.333333,10.0,0.0,0,none,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## 11. Feature table (A6)

In [14]:
feature_table = pd.DataFrame([
    ("hour", "datetime.hour", "pickup_hour", "strong daily profile", "yes"),
    ("day_of_week", "datetime.dayofweek", "pickup_hour", "weekday vs weekend shapes", "yes"),
    ("is_weekend", "day_of_week >= 5", "pickup_hour", "weekend collapse/boost", "yes"),
    ("is_public_holiday", "date in public_holiday events", "events", "holidays change demand", "yes"),
    ("is_school_break", "date in school_break events", "events", "fewer school/commute trips", "yes"),
    ("is_payday_window", "day >= 25 or day <= 3", "pickup_hour", "pay-period spending", "yes"),
    ("days_since_start", "(datetime - 2025-01-01) in days", "pickup_hour", "captures growth trend", "yes"),
    ("hour_sin / hour_cos", "sin/cos(2π·hour/24)", "pickup_hour", "cyclic hour for linear models", "yes"),
    ("rain_mm", "hourly rain, sentinel removed", "weather", "rain shifts walkers to rides", "yes (forecast)"),
    ("rain_3h_sum", "rolling 3h sum of rain_mm", "weather", "lingering rain effect", "yes (forecast)"),
    ("rain_class", "none/light(<2.5)/moderate(<7.6)/heavy", "weather", "non-linear dose response", "yes (forecast)"),
    ("temp_c, humidity_pct, wind_kmh", "cleaned, °F fixed, interpolated", "weather", "comfort effects", "yes (forecast)"),
    ("event_in_window", "zone-hour in [start-2h, end+2h]", "events", "crowds arrive/leave", "yes (scheduled)"),
    ("event_pre / event_during / event_post", "phase of window", "events", "post-event surge", "yes (scheduled)"),
    ("football/concert/conference/road_closure_window", "type flags", "events", "type-specific effects", "yes (scheduled)"),
    ("event_attendance", "max parsed attendance in window", "events", "bigger crowd → more trips", "yes (scheduled)"),
    ("lag_336h, lag_504h", "trips 14/21 days earlier, same zone-hour", "trips", "recent level per zone-hour", "yes (≥ horizon)"),
    ("roll_mean_168h_lag336", "mean of 168 h ending 336 h earlier", "trips", "recent zone level / trend", "yes (≥ horizon)"),
    ("avg_fare_birr, avg_wait_min, active_drivers", "raw", "trips", "consequences of demand", "NO – excluded"),
], columns=["feature", "formula", "source", "why_it_helps", "known_at_forecast_time"])
display(feature_table)

,feature,formula,source,why_it_helps,known_at_forecast_time
0,hour,datetime.hour,pickup_hour,strong daily profile,yes
1,day_of_week,datetime.dayofweek,pickup_hour,weekday vs weekend shapes,yes
2,is_weekend,day_of_week >= 5,pickup_hour,weekend collapse/boost,yes
3,is_public_holiday,date in public_holiday events,events,holidays change demand,yes
4,is_school_break,date in school_break events,events,fewer school/commute trips,yes
5,is_payday_window,day >= 25 or day <= 3,pickup_hour,pay-period spending,yes
6,days_since_start,(datetime - 2025-01-01) in days,pickup_hour,captures growth trend,yes
7,hour_sin / hour_cos,sin/cos(2π·hour/24),pickup_hour,cyclic hour for linear models,yes
8,rain_mm,"hourly rain, sentinel removed",weather,rain shifts walkers to rides,yes (forecast)
9,rain_3h_sum,rolling 3h sum of rain_mm,weather,lingering rain effect,yes (forecast)


## 12. Integrity checks (A7)

In [15]:
NON_FEATURES = {"record_id", "row_id", "pickup_hour", "datetime", "trips",
                "avg_fare_birr", "avg_wait_min", "active_drivers"}
LEAKY = {"avg_fare_birr", "avg_wait_min", "active_drivers", "lag_1h", "lag_24h"}

def validate():
    feats_train = [c for c in master_train.columns if c not in NON_FEATURES]
    feats_test = [c for c in master_test.columns if c not in NON_FEATURES]
    checks = {
        "one row per zone-hour (train)": not master_train.duplicated(["zone", "datetime"]).any(),
        "one row per zone-hour (test)": not master_test.duplicated(["zone", "datetime"]).any(),
        "only the 12 canonical zones": set(master_train.zone) | set(master_test.zone) == set(CANONICAL_ZONES),
        "train timestamps within 1 Jan–31 Oct (EAT)": master_train.datetime.between(TRAIN_START, TRAIN_END).all(),
        "test timestamps within 1–14 Nov (EAT)": master_test.datetime.between(TEST_START, TEST_END).all(),
        "test has 4,032 rows in template order": len(master_test) == 4032 and
            master_test.row_id.tolist() == submission_template.row_id.tolist(),
        "no negative / sentinel trips": (master_train.trips >= 0).all(),
        "no sentinel weather values": (master_train.rain_mm >= 0).all() and master_train.temp_c.between(-5, 40).all()
            and (master_test.rain_mm >= 0).all(),
        "no missing weather after join": master_train[WEATHER_FEATS].notna().all().all()
            and master_test[WEATHER_FEATS].notna().all().all(),
        "joins did not change row counts": len(master_train) == n_before_weather and len(master_test) == len(test_raw),
        "train and test share the same feature columns": feats_train == feats_test,
        "no forecast-time-unavailable feature in test": not (set(feats_test) & LEAKY),
        "weather & event features present (Rule 5)": "rain_mm" in feats_test and "event_in_window" in feats_test,
    }
    for name, ok in checks.items():
        print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    return all(checks.values())

assert validate(), "Integrity checks failed"

[PASS] one row per zone-hour (train)
[PASS] one row per zone-hour (test)
[PASS] only the 12 canonical zones
[PASS] train timestamps within 1 Jan–31 Oct (EAT)
[PASS] test timestamps within 1–14 Nov (EAT)
[PASS] test has 4,032 rows in template order
[PASS] no negative / sentinel trips
[PASS] no sentinel weather values
[PASS] no missing weather after join
[PASS] joins did not change row counts
[PASS] train and test share the same feature columns
[PASS] no forecast-time-unavailable feature in test
[PASS] weather & event features present (Rule 5)


## 13. Cleaning log (A1)

In [16]:
cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)
print(cleaning_log_df.file.value_counts())
cleaning_log_df.to_csv(REPORTS_DIR / "A1_cleaning_log.csv", index=False)

,file,columns,issue_type,rows_affected,pct_rows,fix_applied,why
0,ride_demand_train,pickup_hour,mixed timestamp formats (3),38574,45.14,regex-detected explicit formats; +03:00 conver...,avoid day/month swaps and clock drift
1,ride_demand_train,zone,inconsistent zone spelling (55 variants),32391,37.90,case/space normalization + alias map to 12 labels,one key per zone across all tables
2,ride_demand_train,"zone, pickup_hour",duplicate zone-hour rows,846,0.99,kept first occurrence,exactly one row per zone-hour
3,ride_demand_train,trips,sentinel -1 (no reading),675,0.79,"set to NaN, row excluded from training target",-1 trips is impossible
4,ride_demand_train,trips,missing target,837,0.98,row excluded from training target (kept in gri...,cannot learn from unknown target
5,ride_demand_train,trips,data-entry spikes (trips > 4x active drivers),123,0.14,set to NaN,e.g. 640 trips with 56 drivers; real event pea...
6,ride_demand_train,avg_wait_min,sentinel -1,1692,1.98,set to NaN,negative wait impossible (analysis-only column)
7,ride_demand_train,avg_fare_birr,missing values,1269,1.48,left NaN (not a model input),"operational column, analysis only"
8,ride_demand_train,(rows),platform outage – whole city missing,42,0.58,not imputed; hours excluded from training,"no demand was recorded, imputing would invent ..."
9,ride_demand_train,(rows),late-launching zone (Ayat) – no rows before la...,1752,2.00,zone-hours before launch excluded,zone did not exist; not zero demand


file
ride_demand_train    10
events_calendar       7
weather_hourly        6
Name: count, dtype: int64


## 14. Export master tables + data dictionary (A8)

In [17]:
master_train.to_csv(PROCESSED_DIR / "master_train.csv", index=False)
master_test.to_csv(PROCESSED_DIR / "master_test.csv", index=False)
weather_clean.to_csv(PROCESSED_DIR / "weather_hourly_processed.csv", index=False)
events_clean.assign(zones=events_clean.zones.str.join("|")).to_csv(PROCESSED_DIR / "events_processed.csv", index=False)

source = {**{c: "trips" for c in ["record_id", "zone", "datetime", "trips", "avg_fare_birr", "avg_wait_min",
                                  "active_drivers", "row_id", "pickup_hour"]},
          **{c: "weather" for c in WEATHER_FEATS}, **{c: "events" for c in EVENT_FEATS + ["is_public_holiday", "is_school_break"]},
          **{c: "trips (history)" for c in LAG_FEATS}}
desc = dict(zip(feature_table.feature, feature_table.formula))
rows = []
for c in master_train.columns.union(master_test.columns, sort=False):
    rows.append({"column": c,
                 "dtype": str((master_train if c in master_train else master_test)[c].dtype),
                 "source": source.get(c, "calendar (derived from datetime)"),
                 "in_train": c in master_train, "in_test": c in master_test,
                 "description": next((v for k, v in desc.items() if c in k), "identifier / raw / target"),
                 "known_at_forecast_time": c not in {"trips", "avg_fare_birr", "avg_wait_min", "active_drivers"}})
data_dictionary_master = pd.DataFrame(rows)
data_dictionary_master.to_csv(PROCESSED_DIR / "data_dictionary_master.csv", index=False)
display(data_dictionary_master)

,column,dtype,source,in_train,in_test,description,known_at_forecast_time
0,record_id,str,trips,True,False,identifier / raw / target,True
1,zone,str,trips,True,True,identifier / raw / target,True
2,datetime,datetime64[ns],trips,True,True,identifier / raw / target,True
3,trips,float64,trips,True,False,identifier / raw / target,False
4,avg_fare_birr,float64,trips,True,False,raw,False
5,avg_wait_min,float64,trips,True,False,raw,False
6,active_drivers,int64,trips,True,False,raw,False
7,hour,int32,calendar (derived from datetime),True,True,datetime.hour,True
8,day_of_week,int32,calendar (derived from datetime),True,True,datetime.dayofweek,True
9,is_weekend,int64,calendar (derived from datetime),True,True,day_of_week >= 5,True
